# SMS Spam Detection — TF-IDF + Logistic Regression Baseline

Classify SMS messages as spam or ham using classic NLP feature engineering (TF-IDF, message-level signal features) and Logistic Regression, evaluated with precision/recall due to class imbalance.

Dataset: [SMS Spam Collection (UCI)](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)

# Step 1: Receiving Data

In [1]:
import pandas as pd

In [2]:
# split by tabs not commas
df = pd.read_csv('../data/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

# Step 2: EDA

## Class imbalance

* What's the ratio of ham to spam?
* What's the level of imbalance? Is it critical?
* What are we going to do about this in the future?

In [3]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
df['label'].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

With 86.6% ham / 13.4% spam:

* We'll **not use SMOTE** (more suitable for severe imbalance, and was designed for numeric features).
* Instead we'll use **class_weight='balanced'** to add more value to the minority.

## Are spam messages longer or shorter than ham?

In [5]:
# add length column
df['length'] = df['message'].apply(len)

In [6]:
df.groupby('label')['length'].mean()

label
ham      71.482487
spam    138.670683
Name: length, dtype: float64

In [7]:
df.groupby('label')['length'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4825.0,71.482487,58.440652,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.670683,28.873603,13.0,133.0,149.0,157.0,223.0


**Length itself is a useful input feature**.

**Explanation.** The spam message needs to contain to work: a hook, an offer, urgency, usually a call-to-action.

In [8]:
# how much duplicates we have
df.duplicated().sum()

np.int64(403)

In [9]:
df[df.duplicated()].head()

,label,message,length
103,ham,As per your request 'Melle Melle (Oru Minnamin...,160
154,ham,As per your request 'Melle Melle (Oru Minnamin...,160
207,ham,"As I entered my cabin my PA said, '' Happy B'd...",156
223,ham,"Sorry, I'll call later",22
326,ham,No calls..messages..missed calls,32


Dropping duplicates before splitting into train/test because we need the model correctly classify a message it has never seen before not recognizing concrete string.

In [10]:
print("df.shape before removing duplicates:", df.shape)
df = df.drop_duplicates()
print("df.shape after removing duplicates:", df.shape)

df.shape before removing duplicates: (5572, 3)
df.shape after removing duplicates: (5169, 3)


# Step 3: Text preprocessing

1. **Lowercasing.** Words in the upper and lower case mean the same thing, so we collapse words like them into one.

2. **Removing punctuation.** Strip punctuation so 'prize!!!' and 'prize' count as the same word. Shoutiness (!, $) is captured separately as its own features first.

3. **Removing stopwords.** Words like "the," "is," "at," "and" useless.

4. **Tokenization.** Splits cleaned text into a list of words for the vectorizer to count.

Let's create own numeric columns for `num_exclamations`, `has_currency_symbol`, `pct_uppercase` before lowercase/strip punctuation.

In [11]:
import re
import string

# --- Pass 1: extract signal from RAW text, before cleaning destroys it ---
df['num_exclamations'] = df['message'].apply(lambda x: x.count('!'))
df['has_currency_symbol'] = df['message'].apply(lambda x: 1 if re.search(r'[$£€]', x) else 0)
df['pct_uppercase'] = df['message'].apply(  # raw counts are biased by length; proportions normalize for it
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
)
# tells you "how many digit characters total" but can't distinguish where they are or what pattern they form
df['num_digits'] = df['message'].apply(lambda x: sum(1 for c in x if c.isdigit()))

# --- Pass 2: clean the text for the vectorizer ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove digits (already captured as num_digits)
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # collapse extra whitespace
    return text

df['clean_message'] = df['message'].apply(clean_text)

In [12]:
df.head()

,label,message,length,num_exclamations,has_currency_symbol,pct_uppercase,num_digits,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",111,0,0,0.027027,0,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,29,0,0,0.068966,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,0,0.064516,25,free entry in a wkly comp to win fa cup final ...
3,ham,U dun say so early hor... U c already then say...,49,0,0,0.040816,0,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,0,0,0.032787,0,nah i dont think he goes to usf he lives aroun...


In [13]:
df[['message','clean_message']].sample(5)

,message,clean_message
618,For my family happiness..,for my family happiness
1618,Did u download the fring app?,did u download the fring app
1768,"K, want us to come by now?",k want us to come by now
839,We tried to contact you re our offer of New Vi...,we tried to contact you re our offer of new vi...
511,"8 at the latest, g's still there if you can sc...",at the latest gs still there if you can scroun...


5. **Stemming.** Crudely chops word endings (winning → win).

In [14]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    words = text.split()  # split cleaned text into a list of words
    words = [w for w in words if w not in stop_words]  # keep only non-stopwords
    words = [stemmer.stem(w) for w in words]  # reduce each word to its stem
    return ' '.join(words)  # join back into a single string

df['final_message'] = df['clean_message'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to /home/philip/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
df[['clean_message', 'final_message']].sample(5)

,clean_message,final_message
3932,nooooooo im gonna be bored to death all day ca...,nooooooo im gonna bore death day cabl internet...
2629,haha they cant what at the most tmr forfeit ha...,haha cant tmr forfeit haha
3130,haha better late than ever any way i could swi...,haha better late ever way could swing
5049,yeah so basically any time next week you can g...,yeah basic time next week get away mom amp get
4382,mathews or tait or edwards or anderson,mathew tait edward anderson


**Observation.** Punctuation removal before tokenization can create small side effects like: "that's" -> new token "thats".

# Step 4: TF-IDF

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF (term frequency): how often the word appears within this one message
# IDF (inverse document frequency): weight of the word across the whole dataset
# low terms frequency -> high IDF, high freq. -> low IDF

tfidf = TfidfVectorizer(max_features=3000)  # only the words that show up often enough to be statistically useful
# X_tfidf, is a matrix: one row per message, one column per vocabulary word, filled with TF-IDF scores
X_tfidf = tfidf.fit_transform(df['final_message'])  # it's a sparse matrix

In [17]:
X_tfidf.shape

(5169, 3000)

# Combining TF-IDF with numeric features and **split data**

In [18]:
from scipy.sparse import hstack
import numpy as np

numeric_features = df[['num_exclamations', 'has_currency_symbol', 'pct_uppercase', 'num_digits']].values
X_combined = hstack([X_tfidf, numeric_features])

In [19]:
from sklearn.model_selection import train_test_split

y = df['label'].map({'ham': 0, 'spam': 1})  # scikit-learn's models expect numeric labels

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

In [20]:
print("X_train.shape:", X_train.shape)
print("X_test.shape:", X_test.shape)

X_train.shape: (4135, 3004)
X_test.shape: (1034, 3004)


In [21]:
y_train.value_counts(normalize=True)

label
0    0.873761
1    0.126239
Name: proportion, dtype: float64

In [22]:
y_test.value_counts(normalize=True)

label
0    0.873308
1    0.126692
Name: proportion, dtype: float64

# Step 5: Train Logistic Regression

In [23]:
# Handling imbalance using class_weight='balanced' 
# weigh mistakes on the minority class (spam) more heavily
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [24]:
# Now evaluate it properly
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       903
        spam       0.96      0.92      0.94       131

    accuracy                           0.98      1034
   macro avg       0.97      0.96      0.96      1034
weighted avg       0.98      0.98      0.98      1034

[[898   5]
 [ 11 120]]


Our classes are separable in feature space, and the model can be **sensitive to the minority class and stay precise at the same time**.

# Step 6: Error Analysis

In [25]:
# Get the original test messages back (we need the index)
test_indices = y_test.index
errors = df.loc[test_indices].copy()
errors['predicted'] = y_pred
errors['true_label'] = y_test.values

false_positives = errors[(errors['label'] == 'ham') & (errors['predicted'] == 1)]
false_negatives = errors[(errors['label'] == 'spam') & (errors['predicted'] == 0)]

pd.set_option('display.max_colwidth', None)
print("FALSE POSITIVES (real messages wrongly flagged as spam):")
print(false_positives[['message']])
print("\nFALSE NEGATIVES (spam that slipped through):")
print(false_negatives[['message']])

FALSE POSITIVES (real messages wrongly flagged as spam):
                                                                                                                                                             message
4006                                                                                                         , ow u dey.i paid 60,400thousad.i told  u would call . 
1470         7 wonders in My WORLD 7th You 6th Ur style 5th Ur smile 4th Ur Personality 3rd Ur Nature 2nd Ur SMS and 1st "Ur Lovely Friendship"... good morning dear
1852                                                                                                             Dunno da next show aft 6 is 850. Toa payoh got 650.
4870  1. Tension face 2. Smiling face 3. Waste face 4. Innocent face 5.Terror face 6.Cruel face 7.Romantic face 8.Lovable face 9.decent face  &lt;#&gt; .joker face.
3044                                                                                                                  

* Every single false positive contains digits, and several contain currency symbols.
* False positives messages don't use loud, aggressive style. It's more casual/conversational. Explicit numeric features were built to catch that loud style ("FREE!!! CALL NOW $$$"). The only thing that could have caught them was the TF-IDF.

In [26]:
# how many total occurrences "XXX" actually are
df[df['final_message'].str.contains('xxx')]['label'].value_counts()

label
ham     42
spam    21
Name: count, dtype: int64

## Quick check on the threshold

In [27]:
from sklearn.metrics import precision_recall_curve

# returns two probabilities per message (P(ham), P(spam)) instead of a hard 0/1 label. [:, 1] grabs just the spam probability column.
y_probs = model.predict_proba(X_test)[:, 1]  # probability of spam for each test message
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

for p, r, t in zip(precisions[::10], recalls[::10], thresholds[::10]):
    print(f"threshold={t:.2f}  precision={p:.3f}  recall={r:.3f}")

threshold=0.01  precision=0.127  recall=1.000
threshold=0.01  precision=0.128  recall=1.000
threshold=0.02  precision=0.129  recall=1.000
threshold=0.02  precision=0.130  recall=1.000
threshold=0.02  precision=0.132  recall=1.000
threshold=0.02  precision=0.133  recall=1.000
threshold=0.02  precision=0.134  recall=1.000
threshold=0.02  precision=0.136  recall=1.000
threshold=0.02  precision=0.137  recall=1.000
threshold=0.02  precision=0.139  recall=1.000
threshold=0.02  precision=0.140  recall=1.000
threshold=0.02  precision=0.142  recall=1.000
threshold=0.02  precision=0.143  recall=1.000
threshold=0.02  precision=0.145  recall=1.000
threshold=0.03  precision=0.147  recall=1.000
threshold=0.03  precision=0.148  recall=1.000
threshold=0.03  precision=0.150  recall=1.000
threshold=0.03  precision=0.152  recall=1.000
threshold=0.03  precision=0.153  recall=1.000
threshold=0.03  precision=0.155  recall=1.000
threshold=0.03  precision=0.157  recall=1.000
threshold=0.03  precision=0.159  r

Default 0.5 treshold already sits near the best precision/recall (0.34–0.56). Keeping default.

## Step 7: Streamlit Deployment

We'll use logistic regression fore the fast, reliable deployment (small-scale model and product).

In [28]:
import joblib

joblib.dump(model, '../models/logreg_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

['../models/tfidf_vectorizer.pkl']

In [29]:
!ls

01_baseline_tfidf_logreg.ipynb	02_transformer_comparison.ipynb  results
